# Lakebase work references (oxjob #2347)

`GET /works/{id}/references` reads each work's reference list from Lakebase (`openalex-lookups`). `references_on_works` is
rebuilt every day, so this keeps a MERGE-updated copy the synced tables can follow: `openalex.works.lakebase_work_references_{0,1}`
(shard = `pmod(work_id, 2)`, about 0.5 TB of JSON each; synced tables want under 1 TB), one row per work with a reference list:
`doc` is the API response as served (`{"references": [{"raw_reference_string", "raw_doi", "work_id"}]}`, full URLs, nulls
kept), `doc_hash` the hash of the list. Only works whose list changed, appeared or went are written, so a night writes about
1-3M rows, not 171M. A work with no row has no references (the API answers `[]` when the work exists).

The first run on empty tables is the full build (171.6M rows). Then create the synced tables `openalex.lakebase.lakebase_work_references_{0,1}`
(TRIGGERED, primary key `work_id`) and `GRANT SELECT` them to `openalex_api_ro`; the job's next task triggers their refresh.

Revert: delete this task and the sync task from jobs/references_on_works.yaml, drop the synced tables, then these tables.

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.lakebase_work_references_0 (
  work_id BIGINT NOT NULL, doc STRING, doc_hash BIGINT, updated_at TIMESTAMP,
  CONSTRAINT lakebase_work_references_0_pk PRIMARY KEY (work_id)
) CLUSTER BY (work_id)
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
COMMENT 'oxjob #2347: GET /works/{id}/references docs (one per work with a reference list), for the Lakebase synced table'

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.works.lakebase_work_references_1 (
  work_id BIGINT NOT NULL, doc STRING, doc_hash BIGINT, updated_at TIMESTAMP,
  CONSTRAINT lakebase_work_references_1_pk PRIMARY KEY (work_id)
) CLUSTER BY (work_id)
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
COMMENT 'oxjob #2347: GET /works/{id}/references docs (one per work with a reference list), for the Lakebase synced table'

In [ ]:
CREATE OR REPLACE TABLE openalex.works._lakebase_work_references_changes
TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '7 days', 'delta.logRetentionDuration' = '7 days')
AS
WITH cur AS (SELECT work_id, `references`, xxhash64(`references`) AS h FROM openalex.works.references_on_works),
old AS (
  SELECT work_id, doc_hash FROM openalex.works.lakebase_work_references_0
  UNION ALL SELECT work_id, doc_hash FROM openalex.works.lakebase_work_references_1
)
SELECT coalesce(c.work_id, o.work_id) AS work_id, c.work_id IS NULL AS gone, c.h AS doc_hash,
  CASE WHEN c.work_id IS NOT NULL THEN to_json(named_struct('references', transform(c.`references`, x -> named_struct(
    'raw_reference_string', x.raw_reference_string, 'raw_doi', x.raw_doi,
    'work_id', 'https://openalex.org/W' || x.work_id))), map('ignoreNullFields', 'false')) END AS doc
FROM cur c FULL OUTER JOIN old o ON o.work_id = c.work_id
WHERE c.work_id IS NULL OR o.work_id IS NULL OR c.h <> o.doc_hash

In [ ]:
-- A night drops a few thousand works' lists; millions means references_on_works came out broken. Fails before any write.
SELECT count(*) AS changes, count_if(NOT gone) AS upserts, count_if(gone) AS gone,
  CASE WHEN count_if(gone) > 5000000
    THEN raise_error(concat('lakebase_work_references: ', count_if(gone), ' works would lose their list (> 5M); check references_on_works'))
  END AS failed
FROM openalex.works._lakebase_work_references_changes

In [ ]:
MERGE INTO openalex.works.lakebase_work_references_0 AS t
USING (SELECT * FROM openalex.works._lakebase_work_references_changes WHERE pmod(work_id, 2) = 0) AS c
ON t.work_id = c.work_id
WHEN MATCHED AND c.gone THEN DELETE
WHEN MATCHED THEN UPDATE SET t.doc = c.doc, t.doc_hash = c.doc_hash, t.updated_at = current_timestamp()
WHEN NOT MATCHED AND NOT c.gone THEN INSERT (work_id, doc, doc_hash, updated_at)
  VALUES (c.work_id, c.doc, c.doc_hash, current_timestamp())

In [ ]:
MERGE INTO openalex.works.lakebase_work_references_1 AS t
USING (SELECT * FROM openalex.works._lakebase_work_references_changes WHERE pmod(work_id, 2) = 1) AS c
ON t.work_id = c.work_id
WHEN MATCHED AND c.gone THEN DELETE
WHEN MATCHED THEN UPDATE SET t.doc = c.doc, t.doc_hash = c.doc_hash, t.updated_at = current_timestamp()
WHEN NOT MATCHED AND NOT c.gone THEN INSERT (work_id, doc, doc_hash, updated_at)
  VALUES (c.work_id, c.doc, c.doc_hash, current_timestamp())